In [1]:
from turtle import colormode
from imutils import paths
import imutils
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import timeit
from datetime import datetime
from random import shuffle

from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.utils import class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, VGG19, InceptionResNetV2, DenseNet121, ResNet50V2, ResNet101V2, Xception, InceptionV3
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import losses

import tensorflow_addons as tfa

2022-09-27 20:59:58.623444: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1


In [3]:
path = r'/home/bishal/Research/Allergic-Rhinitis/Dataset/rotate'
dirList = {}
imagePaths = list(paths.list_images(path))
imagePaths

['/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/1/28_1_00003_left.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/1/33_1_00002_right.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/1/33_1_00005_left.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n32_3_00004_left.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n29_1_00004_left.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n32_4_00004_left.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n32_3_00003_right.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n29_3_00002_right.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n32_2_00001_right.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset/non_rotate/dataset3/0/n29_3_00005_left.tif',
 '/home/bishal/Research/Allergic-Rhinitis/Dataset

In [6]:
data = []
labels = []
dirList= []

In [7]:
# Formatting data and labels
for imagePath in imagePaths:
    # Extract the class label from file name and append to labels
    # 파일 이름에서 클래스 레이블을 추출하고 레이블에 추가함
    label = imagePath.split(os.path.sep)[-2]
    #if label=="2":
     #   labels.append("1")
    #else:
     #   labels.append(label)
    labels.append(label)
    # Load the image, swap color channels, and resize it to be a fixed 224x224 pixels while ignoring the aspect ratio
    # 이미지를 로드하고, 컬러 채널을 스왑하고, 가로 세로 비율을 무시하고 고정 224x224 픽셀로 크기를 조정함
    image = cv2.imread(imagePath)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224,224))
    # Append to data
    # 데이터에 추가
    data.append(image)

# Convert the data and labels to NumPy arrays while scaling the pixel intensities to the range [0,1]
# 픽셀 강도를 [0,1] 범위로 조정하면서 데이터와 레이블을 NumPy 배열로 변환
data = np.array(data) / 255.0
labels = np.array(labels)

In [11]:
labels.shape

(90,)

In [21]:
class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
sample_weight = class_weight.compute_sample_weight(class_weight='balanced', y=labels)
print(class_weights)
sample_weight

[0.68181818 1.875      1.        ]


array([1.875     , 1.875     , 1.875     , 0.68181818, 0.68181818,
       0.68181818, 0.68181818, 0.68181818, 0.68181818, 0.68181818,
       0.68181818, 0.68181818, 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.875     , 1.875     ,
       1.875     , 0.68181818, 0.68181818, 0.68181818, 0.68181818,
       0.68181818, 0.68181818, 0.68181818, 0.68181818, 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.875     , 1.875     , 1.875     , 1.875     , 0.68181818,
       0.68181818, 0.68181818, 0.68181818, 0.68181818, 0.68181818,
       0.68181818, 0.68181818, 0.68181818, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.875     ,
       1.875     , 1.875     , 0.68181818, 0.68181818, 0.68181818,
       0.68181818, 0.68181818, 0.68181818, 0.68181818, 0.68181818,
       0.68181818, 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.875     , 1.875     , 1.875  

In [15]:
[np.unique(labels)]

[array(['0', '1', '2'], dtype='<U1')]

In [ ]:
compute_sample_weight